# Cleaning 1.1 - Clean professor and contact person names

This notebook does the following:
    (1) Creates or updates an excel workbook with unique Professor/Contact person combinations for manual review
    (2) Merges the manually cleaned professor_clean and responsible_person columns back onto the individual dataset

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font
import os

In [2]:
# Load data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_0.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## Create or update professor/contact cleaning workbook

In [3]:
# Key columns identifying a professor/contact person combination
key_cols = ["Professor", 
            "Email", 
            "Contact person (if different)", 
            "Contact email (if different)", 
            "Comments?"]
clean_cols = ["professor_clean", 
              "responsible_person", 
              "responsible_person_surname", 
              "responsible_person_firstname"]

# Unique combinations of professor/contact person in the current data
labs_unique = labs[key_cols].drop_duplicates().reset_index(drop=True)

In [4]:
# File and sheet name for the professor/contact cleaning workbook
file_name = config.CLEANING_WORKBOOKS / "professor_contact_cleaning.xlsx"
sheet_name = "Professor_Contact"

# If the workbook doesn't exist yet, create it with all current unique combinations
if not file_name.exists():

    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

    all_cols = key_cols + clean_cols

    # Write header
    for col_idx, col_name in enumerate(all_cols, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font = Font(bold=True)
    ws.freeze_panes = "A2"

    labs_unique_out = labs_unique.copy()
    for c in clean_cols:
        labs_unique_out[c] = "" # leave clean cols blank for manual review
    labs_unique_out = labs_unique_out.fillna("") # replace with empty for better excel display

    # Write data
    for row_idx, row_values in enumerate(labs_unique_out[all_cols].values, start=2):
        for col_idx, value in enumerate(row_values, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    # Adjust column widths
    for col in ["A", "B", "C", "D"]:
        ws.column_dimensions[col].width = 40

    wb.save(file_name)
    print(f"Created cleaning workbook with {len(labs_unique_out)} unique combination(s).")

# If the workbook already exists, only append new combinations
else:

    wb = load_workbook(file_name)
    ws = wb[sheet_name]

    existing = pd.DataFrame(ws.values)
    existing.columns = existing.iloc[0] # Set the first row as variable names
    existing = existing[1:].reset_index(drop=True) # Remove the header row from the data
    existing = existing.replace("", np.nan)

    # Identify combinations not already in the workbook
    merged = labs_unique.merge(existing[key_cols], on=key_cols, how="left", indicator=True)
    new_combos = merged[merged["_merge"] == "left_only"][key_cols]

    if not new_combos.empty:
        new_rows = new_combos.copy()
        for c in clean_cols:
            new_rows[c] = ""
        new_rows = new_rows.fillna("")

        for row in new_rows.values:
            ws.append(list(row))

        wb.save(file_name)
        print(f"Added {len(new_rows)} new combination(s) to the cleaning workbook.")
    else:
        print("No new combinations found.")

No new combinations found.


## Merge cleaned values back onto individual dataset

Only run this after manually filling in `professor_clean` and `responsible_person` in the cleaning workbook.

In [5]:
# Read back cleaned values and merge onto main dataset
cleaned = pd.read_excel(file_name, sheet_name=sheet_name)
cleaned = cleaned.replace("", np.nan)

labs = labs.merge(cleaned[key_cols + clean_cols], on=key_cols, how="left")

## Save processed dataset

In [6]:
# Save processed dataset
labs.to_csv(config.PROCESSED_DATA / "individual_processed_1.csv", index = False)